# Meridian — Return Head Training (Colab)

Trains the `MeridianR` return-prediction head on top of the pre-trained backbone.
Checkpoints save to Google Drive every 5000 steps. Re-run all cells to resume after a session drop.

**Run this before `colab_portfolio_sharpe.ipynb`.**

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
HF_USERNAME   = 'Verdenroz'
HF_TOKEN      = ''            # HF token with read access (required if repos are private)
                              # create at: huggingface.co/settings/tokens
HF_MODEL_REPO = f'{HF_USERNAME}/meridian-checkpoints'
HF_DATA_REPO  = f'{HF_USERNAME}/meridian-data'

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/meridian'
os.makedirs(f'{DRIVE_DIR}/checkpoints/pretrain',  exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints/portfolio', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/mmaps',            exist_ok=True)
print('Drive ready:', DRIVE_DIR)

In [ ]:
# ── 2. HuggingFace login + install deps ───────────────────────────────────────
import subprocess, sys
subprocess.run([
    'pip', 'install', '-q',
    'numpy>=2.1', 'pandas>=2.2', 'pyyaml>=6.0',
    'hmmlearn>=0.3', 'faiss-cpu>=1.8', 'tokenizers>=0.20',
    'huggingface_hub',
], check=True)

from huggingface_hub import login, snapshot_download, hf_hub_download
if HF_TOKEN:
    login(token=HF_TOKEN)
print('HuggingFace ready.')

In [ ]:
# ── 3. Download training code ─────────────────────────────────────────────────
import os, shutil

if not os.path.exists('/content/meridian'):
    local = snapshot_download(repo_id=HF_MODEL_REPO, allow_patterns='code/**')
    shutil.copytree(os.path.join(local, 'code'), '/content/meridian')
    print('Code downloaded.')
else:
    print('Code already present.')

In [ ]:
# ── 4. Download data (cached to Drive across sessions) ────────────────────────
import os, shutil

data_dir  = f'{DRIVE_DIR}/data/mmaps'
train_dir = f'{data_dir}/train'
has_data  = os.path.exists(train_dir) and any(f.endswith('.mmap') for f in os.listdir(train_dir))

if not has_data:
    snapshot_download(repo_id=HF_DATA_REPO, repo_type='dataset', local_dir=data_dir)
    # If data was uploaded with a path prefix (e.g. hf upload ... data/mmaps data/mmaps),
    # snapshot_download nests it — flatten to the expected structure.
    for prefix in [os.path.join(data_dir, 'data', 'mmaps'), os.path.join(data_dir, 'mmaps')]:
        if os.path.exists(os.path.join(prefix, 'train')):
            for item in os.listdir(prefix):
                src = os.path.join(prefix, item)
                dst = os.path.join(data_dir, item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(os.path.join(data_dir, prefix.replace(data_dir + '/', '').split('/')[0]), ignore_errors=True)
            break
    print(f'Data downloaded → {data_dir}')
else:
    print('Data already in Drive.')

In [ ]:
# ── 5. Copy data to local SSD for fast I/O ───────────────────────────────────
import shutil, os

local_data = '/content/data/mmaps'
if not os.path.exists(f'{local_data}/train'):
    print('Copying data from Drive to local SSD (one-time per session)...')
    shutil.copytree(f'{DRIVE_DIR}/data/mmaps', local_data)
    print('Done.')
else:
    print('Local data already present.')

In [ ]:
# ── 6. Download pretrain checkpoint (cached to Drive) ─────────────────────────
import os, shutil

pretrain_ckpt = f'{DRIVE_DIR}/checkpoints/pretrain/backbone_pretrain.pt'
if not os.path.exists(pretrain_ckpt):
    path = hf_hub_download(repo_id=HF_MODEL_REPO, filename='pretrain/backbone_pretrain.pt')
    shutil.copy(path, pretrain_ckpt)
    print(f'Checkpoint downloaded → {pretrain_ckpt}')
else:
    print('Pretrain checkpoint already in Drive.')

In [ ]:
# ── 7. Auto-detect GPU ────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), 'No GPU — enable it in Runtime → Change runtime type.'

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  ({vram_gb:.0f} GB VRAM)')

# Per-asset GDN state (B, N, H, Dh, Dv) is large — tune batch conservatively.
GRAD_CKPT = True
if vram_gb >= 75:    # A100 80GB, H100
    BATCH_SIZE = 4
elif vram_gb >= 35:  # A100 40GB
    BATCH_SIZE = 2
elif vram_gb >= 22:  # L4 22.5GB
    BATCH_SIZE = 1
else:                # V100 16GB, T4 16GB
    BATCH_SIZE = 1

print(f'batch={BATCH_SIZE}  gradient_checkpointing={GRAD_CKPT}')

In [ ]:
# ── 8. Train return head (auto-resumes from Drive) ────────────────────────────
import os, sys, subprocess

CKPT_PORT   = f'{DRIVE_DIR}/checkpoints/portfolio'
BACKBONE    = f'{DRIVE_DIR}/checkpoints/pretrain/backbone_pretrain.pt'
return_best = f'{CKPT_PORT}/return_head_best.pt'

resume_from = return_best if os.path.exists(return_best) else BACKBONE
print(f'Resuming from: {resume_from}')

cmd = [
    sys.executable, '-u',
    '-m', 'train.portfolio_train',
    '--head',            'return',
    '--data-dir',        '/content/data/mmaps',   # local SSD — much faster than Drive
    '--steps',           '50000',
    '--batch-size',      str(BATCH_SIZE),
    '--seq-len',         '64',
    '--lr',              '3e-5',
    '--checkpoint-dir',  CKPT_PORT,
    '--resume-backbone', resume_from,
    '--workers',         '2',
    '--compile',
]
if GRAD_CKPT:
    cmd.append('--gradient-checkpointing')

env = os.environ.copy()
env['PYTHONPATH']       = '/content/meridian'
env['PYTHONUNBUFFERED'] = '1'

print('Running:', ' '.join(cmd))
print('-' * 80)

# Stream output line-by-line
process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Training failed with return code {process.returncode}')
print('Training completed.')